# Estimating Firm Productivity Dynamics with System GMM

Based on Nesheim (ECON0060) lecture notes, when the dependent variable (TFP) is highly persistent, standard Arellano-Bond (1991) Difference GMM can suffer from weak instruments. **Blundell & Bond (1998)** propose System GMM, which adds a level equation with differenced instruments to significantly improve efficiency.

This notebook uses `pydynpd` to estimate our 3-factor production model:
$$TFP_{i,t} = \rho TFP_{i,t-1} + \beta_1 Capital_{i,t} + \beta_2 Labour_{i,t} + \gamma PeerTFP_{i,t} + \alpha_i + \gamma_t + \epsilon_{i,t}$$

In [ ]:
# Install dependencies if needed:
# !pip install "numpy<2.0.0" "pandas<2.2.0" pydynpd

import pandas as pd
import numpy as np
from pydynpd import regression

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

if np.__version__ >= "2.0.0":
    raise ImportError("NumPy version must be less than 2.0.0")
if pd.__version__ >= "2.2.0":
    raise ImportError("Pandas version must be less than 2.2.0")
print("Libraries loaded successfully.")

NumPy version: 1.26.4
Pandas version: 2.1.4
Libraries loaded successfully.


: 

### 1. Generating Synthetic `working_yearly` Panel Data

We generate a balanced panel of firms to mimic the `working_yearly` schema. We intentionally build in a dynamic AR(1) process for TFP to ensure the lagged dependent variable has predictive power alongside firm fixed effects ($
\alpha_i$).

In [2]:
np.random.seed(12345)

# Dimensions for our mock panel
N_firms = 1000
T_years = 10

# Create multi-index structure
firm_ids = np.repeat(np.arange(1, N_firms + 1), T_years)
years = np.tile(np.arange(2010, 2010 + T_years), N_firms)

df = pd.DataFrame({'registered_number': firm_ids, 'year': years})

# Simulate firm fixed effects (unobserved time-invariant quality)
firm_fe = np.repeat(np.random.normal(0, 1, N_firms), T_years)

# Simulate independent factors
df['capital'] = np.random.normal(5, 2, len(df))
df['labor'] = np.random.normal(10, 3, len(df))
df['peer_tfp_ttwa_donut'] = np.random.normal(2, 0.5, len(df)) # Spatial peer effect

# Dynamic AR(1) simulation:
# TFP_t = 0.6 * TFP_t-1 + 0.3 * Capital + 0.4 * Labor + 0.5 * Peer_TFP + Firm_FE + e_it
tfp_list = []
for i in range(N_firms):
    tfp_firm = np.zeros(T_years)
    for t in range(T_years):
        error = np.random.normal(0, 0.5)
        if t == 0:
            tfp_firm[t] = firm_fe[i * T_years] + error
        else:
            tfp_firm[t] = (0.6 * tfp_firm[t-1] + 
                           0.3 * df.loc[i * T_years + t, 'capital'] + 
                           0.4 * df.loc[i * T_years + t, 'labor'] + 
                           0.5 * df.loc[i * T_years + t, 'peer_tfp_ttwa_donut'] + 
                           firm_fe[i * T_years] + error)
    tfp_list.extend(tfp_firm)

df['tfp'] = tfp_list

df.head(15)

,registered_number,year,capital,labor,peer_tfp_ttwa_donut,tfp
0,1,2010,3.032991,6.766955,1.255642,-0.757383
1,1,2011,6.861888,6.386395,2.374146,5.078180
2,1,2012,3.376649,9.011420,2.265829,8.544281
3,1,2013,1.339687,11.728197,2.213251,10.628803
4,1,2014,4.722540,9.113965,3.061406,13.086002
5,1,2015,5.668177,13.924158,1.929615,16.394480
6,1,2016,5.977350,8.746602,2.266595,15.746986
7,1,2017,4.643804,8.347793,1.936626,14.382647
8,1,2018,9.244629,11.209206,1.282642,15.961007
9,1,2019,5.122384,10.809191,1.691352,15.763299


### 2. Specifying the `pydynpd` Model (Blundell-Bond 1998)

The syntax for `pydynpd` combines the structural equation with the instrument matrix design:
1. **Structural Variables:** `tfp L1.tfp capital labor peer_tfp_ttwa_donut` 
2. **GMM Instruments:** `gmm(tfp, 2, 4)` specifies lags 2 through 4 of TFP as internal instruments for the differenced/level equations.
3. **Exogenous IVs:** `iv(peer_tfp_ttwa_donut)` treats the spatial peer variable as strictly exogenous.
4. **Time Effects:** Adding `te` includes time (year) dummy indicators.

In [3]:
# 1. Define your variables programmatically
dep_var = "tfp"
lag_dep_var = f"L1.{dep_var}"
standard_factors = ["capital", "labor"]
peer_effect_var = "peer_tfp_ttwa_donut"  # You can easily change this in a loop later

# 2. Build the structural equation 
# Joins the list of standard factors with spaces, and adds the other variables
structural_eq = f"{dep_var} {lag_dep_var} {' '.join(standard_factors)} {peer_effect_var}"

# 3. Build the Instrument Matrix
# GMM instruments for endogenous/predetermined vars
gmm_inst = f"gmm({dep_var}, 2:4) gmm(capital, 2:3)"
# Standard IVs for strictly exogenous vars
iv_inst = f"iv({peer_effect_var})"

# 4. Build the Options
options = "timedumm"

# 5. Concatenate everything using the pydynpd pipe '|' syntax
command_str = f"{structural_eq} | {gmm_inst} {iv_inst} | {options}"

# Let's print it to verify it looks exactly right before running
print("Generated Command String:")
print(command_str)
print("-" * 50)

# Execute estimation
sys_gmm_model = regression.abond(command_str, df, ['registered_number', 'year'])

Generated Command String:
tfp L1.tfp capital labor peer_tfp_ttwa_donut | gmm(tfp, 2:4) gmm(capital, 2:3) iv(peer_tfp_ttwa_donut) | timedumm
--------------------------------------------------
 Dynamic panel-data estimation, two-step system GMM
 Group variable: registered_number                       Number of obs = 8000    
 Time variable: year                                     Min obs per group: 8    
 Number of instruments = 63                              Max obs per group: 8    
 Number of groups = 1000                                 Avg obs per group: 8.00 
+---------------------+------------+---------------------+-------------+-----------+-----+
|         tfp         |   coef.    | Corrected Std. Err. |      z      |   P>|z|   |     |
+---------------------+------------+---------------------+-------------+-----------+-----+
|        L1.tfp       | 0.7687353  |      0.0066456      | 115.6765953 | 0.0000000 | *** |
|       capital       | 0.3267504  |      0.0583535      |  5.599

### 3. How to Interpret Output Diagnostics

When interpreting results generated by `pydynpd`:

1. **Coefficients:** Verify that $L1.tfp$ is positive and statistically significant (indicating persistent dynamics). Check structural signs on input factors (`capital`, `labor`) and the peer externality (`peer_tfp_ttwa_donut`).
2. **Arellano-Bond AR(1) Test:** The null of no first-order serial correlation in first differences *should be rejected* ($p < 0.05$).
3. **Arellano-Bond AR(2) Test:** The null of no second-order correlation *must not be rejected* ($p > 0.05$) to validate instrument lag depth.
4. **Hansen J-Test:** Tests overidentifying restrictions. A $p$-value $> 0.05$ indicates valid, uncorrelated instruments.

In [4]:
# Inspect underlying regression table for export
reg_table = sys_gmm_model.models[0].regression_table
print(reg_table)

# Example of extracting a specific p-value:
# l1_tfp_pvalue = reg_table.loc[reg_table['variable'] == 'L1.tfp', 'p_value'].values[0]
# print(f"P-value for lagged TFP: {l1_tfp_pvalue}")

               variable  coefficient   std_err     z_value        p_value  sig
0                L1.tfp     0.768735  0.006646  115.676595   0.000000e+00  ***
1               capital     0.326750  0.058354    5.599496   2.149755e-08  ***
2                 labor     0.413982  0.003025  136.850978   0.000000e+00  ***
3   peer_tfp_ttwa_donut     0.491793  0.017734   27.731390  2.921863e-169  ***
4             year_2012    -1.097124  0.053810  -20.388812   2.101884e-92  ***
5             year_2013    -1.723834  0.077866  -22.138591  1.343730e-108  ***
6             year_2014    -2.147358  0.093818  -22.888571  6.038763e-116  ***
7             year_2015    -2.409579  0.098964  -24.347990  6.088621e-131  ***
8             year_2016    -2.561928  0.105428  -24.300321  1.945086e-130  ***
9             year_2017    -2.601712  0.107017  -24.311144  1.494505e-130  ***
10            year_2018    -2.718657  0.113105  -24.036562  1.153798e-127  ***
11            year_2019    -2.723395  0.109844  -24.